### Structured output used for to retirve response from model in a structure form (in a specific format)

### There are some different methods to do this
1. pydantic model
2. TypeDict

In [3]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model

model=init_chat_model("groq:openai/gpt-oss-20b")


### Pydantic Model

In [4]:
from pydantic import BaseModel, Field

class University(BaseModel):
    Name: str = Field(description="The name of the university")
    Year_Founded: int = Field(description="The year the university was founded")
    Speciality: str = Field(description="The Speciality of the university")
    City: str = Field(description="The city where the university is located")

structured_model = model.with_structured_output(University)
response = structured_model.invoke("give me the details of ten universitis in Pakisan")
response

University(Name='University of the Punjab', Year_Founded=1882, Speciality='Comprehensive University', City='Lahore')

### Nested Structure

In [ ]:
class courses(BaseModel):
    Name: str = Field(description="The name of course")
    credit_hrs: str =  Field(description="The Credit hours of course")
    code: str = Field(description="Code of Course")

class University(BaseModel):
    Name: str = Field(description="The name of the university")
    course: list[courses]
    Year_Founded: int = Field(description="The year the university was founded")
    Speciality: str = Field(description="The Speciality of the university")
    City: str = Field(description="The city where the university is located")


structured_model =  model.with_structured_output(University, include_raw=True)
response = structured_model.invoke("Can you tell me the details of Universität Paderborn in German")
print(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function to get details. The user wants details of Universität Paderborn in German. So we should call the function with city, name, speciality, year founded, courses. The function expects a JSON with fields City, Name, Speciality, Year_Founded, course: array of objects with Name, code, credit_hrs.\n\nWe need to provide details. Let\'s gather info: Universität Paderborn located in Paderborn, Germany. Founded 1971. Speciality: multi-disciplinary, strong in Computer Science, Engineering, Economics, etc. Provide courses: maybe some typical courses: "Informatik" (CS), "Elektrotechnik" (Electrical Engineering), "Wirtschaftswissenschaften" (Business), "Maschinenbau" (Mechanical Engineering). Provide code and credit hours.\n\nWe need to output in German. So we should call function with the data, but we can also provide the text. The user asked "Can you tell me the details of Universität Paderborn in Germ

### TypeDict (we use it for structed response but when the validation is not concern)

In [7]:
from typing_extensions import TypedDict
class University(TypedDict):
    Name: str = Field(description="The name of the university")
    Year_Founded: int = Field(description="The year the university was founded")
    Speciality: str = Field(description="The Speciality of the university")
    City: str = Field(description="The city where the university is located")

model_withtypedict = model.with_structured_output(University)
response = model_withtypedict.invoke("Please give me the details of Paderborn University")
response

{'City': 'Paderborn',
 'Name': 'Paderborn University',
 'Speciality': 'Engineering, Computer Science, Natural Sciences, Economics, Social Sciences',
 'Year_Founded': 1971}

In [14]:
class courses(TypedDict):
    Name: str
    credit_hrs: str
    code: str

class University(TypedDict):
    Name: str
    course: list[courses]
    Year_Founded: int
    Speciality: str
    City: str


structured_typedictmodel =  model.with_structured_output(University)
response = structured_typedictmodel.invoke("Can you tell me the details of Universität Paderborn in German")
print(response)

{'City': 'Paderborn', 'Name': 'Universität Paderborn', 'Speciality': 'Informatik, Wirtschaftswissenschaften, Elektrotechnik', 'Year_Founded': 1972, 'course': [{'Name': 'Einführung in die Informatik', 'code': 'INF101', 'credit_hrs': '5'}, {'Name': 'Grundlagen der Wirtschaftswissenschaften', 'code': 'WIR201', 'credit_hrs': '4'}, {'Name': 'Elektrische Schaltungen', 'code': 'ELE301', 'credit_hrs': '4'}]}


### Dataclass 

In [ ]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field

class University(BaseModel):
    """Resturn University Response in this format"""
    Name: str = Field(description="The name of the university")
    Year_Founded: int = Field(description="The year the university was founded")
    Speciality: str = Field(description="The Speciality of the university")
    City: str = Field(description="The city where the university is located")


agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    response_format=University
)

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Please give me the details about Paderborn University in Germany."
        }
    ]
})
response["structured_response"]


University(Name='University of Paderborn', Year_Founded=1972, Speciality='Engineering, Computer Science, Economics, Natural Sciences', City='Paderborn')

In [ ]:
from dataclasses import dataclass
from typing_extensions import TypedDict
from langchain.tools import tool

@dataclass
class University(TypedDict):
    """Resturn University Response in this format"""
    Name: str
    Year_Founded: int
    Speciality: str
    City: str


from langchain.tools import tool

@tool
def Get_University_Details(university: str) -> str:
    """Get information about a university.

    Paderborn University is a public research university located
    in Paderborn, Germany. It was founded in 1972 and is known
    for its research and teaching in computer science, information
    technology, engineering, and other academic disciplines.
    """
    return "University information retrieved."

toolmodl = model.bind_tools([Get_University_Details])

response = toolmodl.invoke("Paderborn")
if response is None:
    f"No Information is found"

print(response)


{'input_tokens': 180, 'output_tokens': 55, 'total_tokens': 235, 'output_token_details': {'reasoning': 26}}
